# Hierarchical Transformer with Learned Segmentation

This notebook implements a project-scale research extension of the hierarchical transformer setup. Instead of splitting on whitespace, the model learns latent segment boundaries directly from bytes and uses those segments as the higher-level units for a causal backbone.

In [ ]:
!pip install -q datasets tqdm matplotlib

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

from experiment_utils import count_parameters, pick_device, train_model
from metrics import evaluate_all_metrics, boundary_precision_recall_f1, whitespace_boundary_positions
from results_utils import save_experiment_artifacts


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


@dataclass
class Config:
    train_stories: int = 5000
    val_stories: int = 2000
    byte_block_size: int = 256
    batch_size: int = 16
    byte_d_model: int = 192
    segment_d_model: int = 256
    byte_heads: int = 4
    segment_heads: int = 4
    byte_layers: int = 2
    segment_layers: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    epochs: int = 3
    eval_every: int = 300
    val_max_batches: int = 20
    max_train_steps: Optional[int] = 15000
    boundary_temperature: float = 1.0
    target_boundary_rate: float = 0.12
    boundary_loss_weight: float = 0.1
    sample_prompt: str = "Once upon a time"
    sample_tokens: int = 200


cfg = Config()
cfg

## Design

This extension keeps the hierarchical idea but removes fixed whitespace segmentation. A boundary predictor proposes latent segment starts over the byte sequence, byte states are pooled into segment embeddings, a causal transformer models segment-level context, and the resulting contextualized segment states are broadcast back to bytes for next-byte prediction.

In [ ]:
ds = load_dataset("roneneldan/TinyStories")

train_split = ds["train"].select(range(min(cfg.train_stories, len(ds["train"]))))
val_split = ds["validation"].select(range(min(cfg.val_stories, len(ds["validation"]))))

train_texts = train_split["text"]
val_texts = val_split["text"]

print(f"Train stories: {len(train_texts):,}")
print(f"Validation stories: {len(val_texts):,}")

In [ ]:
BYTE_VOCAB_SIZE = 256


def text_to_bytes(text):
    return list(text.encode("utf-8"))


def bytes_to_text(byte_ids):
    return bytes([int(x) for x in byte_ids if 0 <= int(x) < 256]).decode("utf-8", errors="ignore")


train_text = "\n\n".join(train_texts)
val_text = "\n\n".join(val_texts)

train_ids = torch.tensor(text_to_bytes(train_text), dtype=torch.long)
val_ids = torch.tensor(text_to_bytes(val_text), dtype=torch.long)

print(train_ids.shape, val_ids.shape)

## Dataset

In [ ]:
class ByteBlockDataset(Dataset):
    def __init__(self, byte_ids, block_size):
        self.byte_ids = byte_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.byte_ids) - self.block_size

    def __getitem__(self, idx):
        x = self.byte_ids[idx : idx + self.block_size]
        y = self.byte_ids[idx + 1 : idx + self.block_size + 1]
        return x, y


train_dataset = ByteBlockDataset(train_ids, cfg.byte_block_size)
val_dataset = ByteBlockDataset(val_ids, cfg.byte_block_size)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

print(f"Train windows: {len(train_dataset):,}")
print(f"Validation windows: {len(val_dataset):,}")

## Learned Segmentation Model

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=256):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, block_size, block_size), persistent=False)

    def forward(self, x):
        batch_size, seq_len, channels = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:, :, :seq_len, :seq_len], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        out = self.out_proj(out)
        return self.resid_dropout(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout, block_size=block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class LearnedSegmentationTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.byte_block_size = cfg.byte_block_size
        self.target_boundary_rate = cfg.target_boundary_rate
        self.boundary_loss_weight = cfg.boundary_loss_weight
        self.boundary_temperature = cfg.boundary_temperature

        self.byte_embedding = nn.Embedding(BYTE_VOCAB_SIZE, cfg.byte_d_model)
        self.byte_position_embedding = nn.Embedding(cfg.byte_block_size, cfg.byte_d_model)
        self.byte_blocks = nn.ModuleList(
            [TransformerBlock(cfg.byte_d_model, cfg.byte_heads, dropout=cfg.dropout, block_size=cfg.byte_block_size) for _ in range(cfg.byte_layers)]
        )
        self.byte_ln = nn.LayerNorm(cfg.byte_d_model)

        self.boundary_head = nn.Linear(cfg.byte_d_model, 1)

        self.segment_input_proj = nn.Linear(cfg.byte_d_model, cfg.segment_d_model)
        self.segment_position_embedding = nn.Embedding(cfg.byte_block_size, cfg.segment_d_model)
        self.segment_blocks = nn.ModuleList(
            [TransformerBlock(cfg.segment_d_model, cfg.segment_heads, dropout=cfg.dropout, block_size=cfg.byte_block_size) for _ in range(cfg.segment_layers)]
        )
        self.segment_ln = nn.LayerNorm(cfg.segment_d_model)
        self.segment_output_proj = nn.Linear(cfg.segment_d_model, cfg.byte_d_model)

        self.decoder_fuse = nn.Linear(2 * cfg.byte_d_model, cfg.byte_d_model)
        self.lm_head = nn.Linear(cfg.byte_d_model, BYTE_VOCAB_SIZE)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _sample_boundaries(self, logits):
        probs = torch.sigmoid(logits)

        if self.training:
            uniform = torch.rand_like(logits).clamp_(1e-6, 1 - 1e-6)
            logistic_noise = torch.log(uniform) - torch.log1p(-uniform)
            soft = torch.sigmoid((logits + logistic_noise) / self.boundary_temperature)
            hard = (soft > 0.5).float()
            samples = hard.detach() - soft.detach() + soft
        else:
            samples = (probs > 0.5).float()

        first = torch.ones(logits.size(0), 1, device=logits.device)
        samples = torch.cat([first, samples[:, 1:]], dim=1)
        probs = torch.cat([first, probs[:, 1:]], dim=1)
        return samples, probs

    def _pool_segments(self, byte_states, boundary_samples):
        batch_size, seq_len, dim = byte_states.shape
        segment_ids = boundary_samples.long().cumsum(dim=1) - 1
        num_segments = segment_ids.max(dim=1).values + 1
        max_segments = int(num_segments.max().item())

        segment_embeddings = byte_states.new_zeros(batch_size, max_segments, dim)
        segment_counts = byte_states.new_zeros(batch_size, max_segments, 1)

        for batch_idx in range(batch_size):
            ids = segment_ids[batch_idx]
            segment_embeddings[batch_idx].index_add_(0, ids, byte_states[batch_idx])
            ones = byte_states.new_ones(seq_len, 1)
            segment_counts[batch_idx].index_add_(0, ids, ones)

        segment_embeddings = segment_embeddings / segment_counts.clamp(min=1.0)
        segment_mask = torch.arange(max_segments, device=byte_states.device)[None, :] < num_segments[:, None]
        return segment_embeddings, segment_mask, segment_ids

    def _contextualize_segments(self, segment_embeddings, segment_mask):
        batch_size, max_segments, _ = segment_embeddings.shape
        positions = torch.arange(max_segments, device=segment_embeddings.device)
        x = self.segment_input_proj(segment_embeddings) + self.segment_position_embedding(positions)[None, :, :]

        for block in self.segment_blocks:
            x = block(x)

        x = self.segment_ln(x)
        x = self.segment_output_proj(x)
        x = x * segment_mask[:, :, None]
        return x

    def _expand_segments(self, contextual_segments, segment_ids):
        batch_size, seq_len = segment_ids.shape
        dim = contextual_segments.size(-1)
        expanded = contextual_segments.new_zeros(batch_size, seq_len, dim)

        for batch_idx in range(batch_size):
            expanded[batch_idx] = contextual_segments[batch_idx].index_select(0, segment_ids[batch_idx])

        return expanded

    def forward(self, x, targets=None, return_aux=False):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device)

        byte_states = self.byte_embedding(x) + self.byte_position_embedding(positions)[None, :, :]
        for block in self.byte_blocks:
            byte_states = block(byte_states)
        byte_states = self.byte_ln(byte_states)

        boundary_logits = self.boundary_head(byte_states).squeeze(-1)
        boundary_samples, boundary_probs = self._sample_boundaries(boundary_logits)

        segment_embeddings, segment_mask, segment_ids = self._pool_segments(byte_states, boundary_samples)
        contextual_segments = self._contextualize_segments(segment_embeddings, segment_mask)
        expanded_segments = self._expand_segments(contextual_segments, segment_ids)

        fused_states = torch.cat([byte_states, expanded_segments], dim=-1)
        fused_states = torch.tanh(self.decoder_fuse(fused_states))
        logits = self.lm_head(fused_states)

        loss = None
        lm_loss = None
        boundary_reg = None
        if targets is not None:
            lm_loss = F.cross_entropy(logits.reshape(batch_size * seq_len, BYTE_VOCAB_SIZE), targets.reshape(batch_size * seq_len))
            boundary_reg = (boundary_probs[:, 1:].mean() - self.target_boundary_rate).abs()
            loss = lm_loss + self.boundary_loss_weight * boundary_reg

        if return_aux:
            aux = {
                "boundary_logits": boundary_logits,
                "boundary_probs": boundary_probs,
                "boundary_samples": boundary_samples,
                "segment_ids": segment_ids,
                "num_segments": segment_mask.sum(dim=1),
                "lm_loss": lm_loss,
                "boundary_reg": boundary_reg,
            }
            return logits, loss, aux

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.byte_block_size :]
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < values[:, [-1]]] = float("-inf")

            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)

        return idx

    @torch.no_grad()
    def predict_boundaries_for_text(self, text, device):
        raw_bytes = text_to_bytes(text)
        byte_ids = torch.tensor(raw_bytes, dtype=torch.long, device=device)[None, :]
        if byte_ids.size(1) == 0:
            return set()
        byte_ids = byte_ids[:, : self.byte_block_size]

        byte_to_char = {}
        byte_cursor = 0
        for char_idx, ch in enumerate(text):
            if byte_cursor > self.byte_block_size:
                break
            byte_to_char[byte_cursor] = char_idx
            byte_cursor += len(ch.encode("utf-8"))
        byte_to_char[byte_cursor] = len(text)

        _, _, aux = self(byte_ids, return_aux=True)
        starts = aux["boundary_samples"][0].nonzero(as_tuple=False).squeeze(-1).tolist()
        boundaries = {byte_to_char[pos] for pos in starts[1:] if pos in byte_to_char}
        return boundaries

## Training

In [ ]:
device = pick_device()
print(f"Using device: {device}")

In [ ]:
model = LearnedSegmentationTransformer(cfg).to(device)
print(f"Model parameters: {count_parameters(model) / 1e6:.2f}M")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    checkpoint_path="learned_segmentation_best.pt",
)

history["best_val_loss"]

In [ ]:
if history["steps"]:
    plt.figure(figsize=(10, 4))
    plt.plot(history["steps"], history["train_loss"], label="train loss")
    plt.plot(history["steps"], history["val_loss"], label="val loss")
    plt.xlabel("step")
    plt.ylabel("byte cross-entropy")
    plt.title("Learned Segmentation Training Curve")
    plt.legend()
    plt.show()
else:
    print("No intermediate evaluations were recorded. Lower cfg.eval_every if you want a curve.")

## Inspect Learned Boundaries

In [ ]:
analysis_text = val_texts[0][: cfg.byte_block_size]
model.load_state_dict(torch.load("learned_segmentation_best.pt", map_location=device))
model.eval()

predicted_boundaries = model.predict_boundaries_for_text(analysis_text, device=device)
gold_boundaries = whitespace_boundary_positions(analysis_text)
boundary_scores = boundary_precision_recall_f1(predicted_boundaries, gold_boundaries)

print(analysis_text)
print(boundary_scores)

## Generation

In [ ]:
prompt_ids = torch.tensor([text_to_bytes(cfg.sample_prompt)], dtype=torch.long, device=device)

model.load_state_dict(torch.load("learned_segmentation_best.pt", map_location=device))
sample_ids = model.generate(prompt_ids, max_new_tokens=cfg.sample_tokens, temperature=0.9, top_k=50)
sample_text = bytes_to_text(sample_ids[0].tolist())
print(sample_text)

## Evaluation and Artifact Export

In [ ]:
ood_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ood_texts = [text for text in ood_dataset["text"] if text.strip()][:200]


def byte_nll_fn(texts):
    model.eval()
    total_nll = 0.0
    total_bytes = 0

    with torch.no_grad():
        for text in texts:
            byte_ids = text_to_bytes(text)
            if len(byte_ids) < 2:
                continue

            for start in range(0, len(byte_ids) - 1, cfg.byte_block_size):
                chunk = byte_ids[start : start + cfg.byte_block_size + 1]
                if len(chunk) < 2:
                    continue

                x = torch.tensor(chunk[:-1], dtype=torch.long, device=device)[None, :]
                y = torch.tensor(chunk[1:], dtype=torch.long, device=device)[None, :]
                logits, _ = model(x)
                nll = F.cross_entropy(
                    logits.reshape(-1, BYTE_VOCAB_SIZE),
                    y.reshape(-1),
                    reduction="sum",
                )

                total_nll += nll.item()
                total_bytes += y.numel()

    return total_nll / total_bytes


predicted_boundary_sets = [model.predict_boundaries_for_text(text[: cfg.byte_block_size], device=device) for text in val_texts[:200]]
gold_boundary_sets = [whitespace_boundary_positions(text[: cfg.byte_block_size]) for text in val_texts[:200]]
metrics = evaluate_all_metrics(
    in_domain_texts=val_texts,
    ood_texts=ood_texts,
    byte_nll_fn=byte_nll_fn,
    predicted_boundary_sets=predicted_boundary_sets,
    gold_boundary_sets=gold_boundary_sets,
    max_items=200,
    noisy_char_error_rate=0.05,
    noise_seed=42,
)
metrics

In [ ]:
save_experiment_artifacts(
    "learned_segmentation",
    metrics=metrics,
    config=cfg,
    sample_text=sample_text,
    extra={"boundary_inspection": boundary_scores, "ood_preview": ood_texts[:5]},
)

print("Saved learned_segmentation_metrics.json, learned_segmentation_config.json, learned_segmentation_sample.txt")